<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/04_introduction_to_altair.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Visualization with Altair


# 1.0 Setup and Imports

Ensure necessary libraries are installed. In Google Colab, some common libraries are pre-installed. If running locally or in a different environment, you might need to run:

`!pip install altair vega_datasets polars`


In [1]:

import altair as alt
import polars as pl
from vega_datasets import data as vega_data # For loading sample datasets

# Enable the Altair renderer for Google Colab
# If you're using a different environment (like JupyterLab or a classic Notebook),
# you might need to change this. Examples:
# alt.renderers.enable('jupyterlab')
# alt.renderers.enable('notebook')
# alt.renderers.enable('default')
alt.renderers.enable('colab')

print("Libraries imported and Altair renderer enabled for Colab.")


Libraries imported and Altair renderer enabled for Colab.



### 1.1 Why Visualize Data? Anscombe's Quartet

Summary statistics can be informative, but they don't tell the whole story.

Anscombe's Quartet demonstrates this perfectly.


In [2]:
# Load Anscombe's Quartet from vega_datasets
anscombe_pd_df = vega_data.anscombe() # This loads as a Pandas DataFrame

# Convert to a Polars DataFrame (as we're using Polars in this course)
anscombe_pl_df = pl.from_pandas(anscombe_pd_df)

# Let's inspect the data structure
print("First few rows of Anscombe's Quartet (Polars DataFrame):")
print(anscombe_pl_df.head())

First few rows of Anscombe's Quartet (Polars DataFrame):
shape: (5, 3)
┌────────┬─────┬──────┐
│ Series ┆ X   ┆ Y    │
│ ---    ┆ --- ┆ ---  │
│ str    ┆ i64 ┆ f64  │
╞════════╪═════╪══════╡
│ I      ┆ 10  ┆ 8.04 │
│ I      ┆ 8   ┆ 6.95 │
│ I      ┆ 13  ┆ 7.58 │
│ I      ┆ 9   ┆ 8.81 │
│ I      ┆ 11  ┆ 8.33 │
└────────┴─────┴──────┘


In [10]:

# Now, let's calculate key summary statistics for each dataset within the quartet.
# We'll group by the 'Dataset' column.

summary_stats = anscombe_pl_df.group_by("Series").agg(
    pl.mean("X").alias("Mean_X"),
    pl.std("X").alias("StdDev_X"),
    pl.mean("Y").alias("Mean_Y"),
    pl.std("Y").alias("StdDev_Y"),
    pl.corr("X", "Y").alias("Correlation_XY")
).sort("Series") # Sort for consistent display

print("\nSummary Statistics for Anscombe's Quartet:")
print(summary_stats)



Summary Statistics for Anscombe's Quartet:
shape: (4, 6)
┌────────┬────────┬──────────┬──────────┬──────────┬────────────────┐
│ Series ┆ Mean_X ┆ StdDev_X ┆ Mean_Y   ┆ StdDev_Y ┆ Correlation_XY │
│ ---    ┆ ---    ┆ ---      ┆ ---      ┆ ---      ┆ ---            │
│ str    ┆ f64    ┆ f64      ┆ f64      ┆ f64      ┆ f64            │
╞════════╪════════╪══════════╪══════════╪══════════╪════════════════╡
│ I      ┆ 9.0    ┆ 3.316625 ┆ 7.5      ┆ 2.03289  ┆ 0.816186       │
│ II     ┆ 9.0    ┆ 3.316625 ┆ 7.500909 ┆ 2.031657 ┆ 0.816237       │
│ III    ┆ 9.0    ┆ 3.316625 ┆ 7.5      ┆ 2.030424 ┆ 0.816287       │
│ IV     ┆ 9.0    ┆ 3.316625 ┆ 7.500909 ┆ 2.030579 ┆ 0.816521       │
└────────┴────────┴──────────┴──────────┴──────────┴────────────────┘


**Note: The summary statistics (mean, std dev, correlation) are nearly identical for all four datasets!**

### 1.2 Visualizing Anscombe's Quartet with Altair

 Now, let's see what these datasets *look* like.
 We will create a scatter plot for each dataset.

 **The core Altair syntax: `alt.Chart(data).mark_type().encode(visual_channels)`**

 We specify the data type for X and Y as Quantitative ('Q') using a colon.

 This helps Altair apply appropriate scales and axes.
 Example: `x='X:Q'`


In [24]:
def dataset_mapper(val: str) -> str:
  dataset_map = {
    'I': 'Linear',
    'II': 'Non-linear',
    'III': 'Linear with outlier',
    'IV': 'Vertical line with outlier'
  }
  return dataset_map.get(val, 'Unknown')

anscombe = anscombe_pl_df.select(
    pl.col('X', 'Y'),
    pl.col('Series')
      .map_elements(dataset_mapper, return_dtype=pl.String)
)

(
    alt.Chart(anscombe)
    .mark_point(size=60)
    .encode(
        alt.X('X:Q', scale=alt.Scale(domain=[0, 20])),
        alt.Y('Y:Q', scale=alt.Scale(domain=[0, 14])),
        alt.Color('Series:N')
          .legend(title="Dataset"),
        alt.Facet('Series:N')
        .columns(2)
        .title(None)
    )
    .properties(
        title="Anscombe's Quartet",
        width=200,
        height=200,
    )
)

alt.Chart(...)

How does the visualization compare with the summary statistics?
- Dataset I: Appears to be a linear relationship.
- Dataset II: Shows a clear non-linear (curved) relationship.
- Dataset III: A linear relationship with a significant outlier.
- Dataset IV: Most X values are constant, with one influential outlier.

Visualization helps identify patterns, anomalies, and guides further analysis.



### 1.3 Understanding Altair's Declarative Nature & Core Idea

Recall: `object = data + behavior`

You've just used this syntax to create the Anscombe plots!
- `alt.Chart(anscombe)`: Create a `Chart` object with `anscombe` **data**.
- `.mark_point()`: Ask the chart object to use  `point` as **mark** type.
- `.encode(x='X:Q', y='Y:Q')`: Ask chart to **encode** `X` and `Y` columns
  as **Q**uantitative  data type to visual properties (channels) `x-position` and `y-position`.

Any diagram you create in Altair follow this basic structure. This consistent structure is key to Altair's power and ease of use.
By focusing on what you want to represent, you can build a wide variety of charts.